**Capítulo 8**  
**Base SIMUL074.xlsx**

In [1]:
# Bibliotecas e funções necessárias
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, mean_absolute_percentage_error
import tensorflow as tf
from tensorflow.keras import layers, callbacks, models
import joblib

In [ ]:
# Semente para reprodutibilidade
SEED = 11
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Leitura da base de dados
mm = pd.read_excel("SIMUL074.xlsx")

In [ ]:
# Inicialmente, padronizamos os dados entre 0 e 1 e geramos a matriz de dados padronizados zz :

mini = mm.min(axis=0)
maxi = mm.max(axis=0)
den = (maxi - mini).replace(0, np.nan)

zz = (mm - mini) / den
mini_, maxi_ = mini.copy(), maxi.copy()

print(zz.head().round(3))

In [ ]:
# A amostra será dividida em duas partes, uma para treinamento e a segunda para teste da rede.
SEED = 123
train, test = train_test_split(zz, train_size=1000, shuffle=True, random_state=SEED)

train.shape, test.shape

In [ ]:
# Vamos obter o modelo de regressão linear múltipla.
# Servirá de padrão para avaliar a eficácia preditiva da rede neural

X_train = train.drop(columns=['y']).values
y_train = train['y'].values
X_test  = test.drop(columns=['y']).values
y_test  = test['y'].values

# modelo linear
rm = LinearRegression()
rm.fit(X_train, y_train)

# previsões no teste
rm_pred_tst = rm.predict(X_test)

# métricas
rmse_reg  = np.sqrt(np.mean((y_test - rm_pred_tst)**2))
mape_reg  = np.mean(np.abs((y_test - rm_pred_tst)/y_test)) * 100  

print(f"[Linear] RMSE (norm): {rmse_reg:.8f}")
print(f"[Linear] MAPE (norm): {mape_reg:.5f}%")

In [ ]:
from sklearn.neural_network import MLPRegressor

SEED = 123

# Entradas já definidas anteriormente:
# X_train, y_train, X_test, y_test  

rn1 = MLPRegressor(
    hidden_layer_sizes=(4,),
    activation="relu",
    solver="lbfgs",
    alpha=0.0001,
    max_iter=1000,
    random_state=SEED,
)

rn1.fit(X_train, y_train)

# previsões no teste
rn1_pred_test = rn1.predict(X_test)

# métricas (na escala normalizada)
rmse_nn = np.sqrt(np.mean((y_test - rn1_pred_test) ** 2))
mape_nn = np.mean(np.abs((y_test - rn1_pred_test) / y_test)) * 100

print(f"[NN] convergiu? {rn1.n_iter_ }")
print(f"[NN] RMSE (norm): {rmse_nn:.6f}")
print(f"[NN] MAPE (norm): {mape_nn:.2f}%")

In [ ]:
# transformação inversa 
y_min, y_max = mini_['y'], maxi_['y']
def unscale_y(arr):
    arr = np.asarray(arr)
    return arr * (y_max - y_min) + y_min

# y previsto pela rede (na escala original) e y real (original)
yhat_orig = unscale_y(rn1_pred_test)       
yold_orig = unscale_y(test['y'].values)   

aux = pd.DataFrame({"y": yold_orig, "yhat": yhat_orig})
print("\nhead(aux):")
print(aux.head().round(5))

In [ ]:
# 8.11 Aplicação de uma RNA para classificação

In [2]:
# ler o Excel (ajuste o caminho se necessário)
tec = pd.read_excel("TECAL.xlsx")
tec = tec.iloc[:, 1:].copy()

# dummies
tec_num = pd.get_dummies(tec, drop_first=True).astype(float)

mini = tec_num.min(axis=0)
maxi = tec_num.max(axis=0)
ampl = (maxi - mini).replace(0, np.nan)

tecx = (tec_num - mini) / ampl
mini_, maxi_, ampl_ = mini.copy(), maxi.copy(), ampl.copy()

print(tecx.head().round(3))

   idade  linhas  temp_cli  renda  fatura  temp_rsd  local_B  local_C  \
0  0.737     1.0     0.368  0.043   0.134     0.562      0.0      0.0   
1  0.342     0.5     0.105  0.055   0.119     0.344      0.0      0.0   
2  0.316     0.0     0.079  0.025   0.146     0.367      0.0      0.0   
3  0.553     0.0     0.105  0.046   0.160     0.484      0.0      1.0   
4  0.447     0.0     0.263  0.092   0.294     0.477      0.0      1.0   

   local_D  tvcabo_sim  debaut_sim  cancel_sim  
0      0.0         1.0         0.0         0.0  
1      0.0         1.0         0.0         0.0  
2      0.0         0.0         0.0         0.0  
3      0.0         1.0         1.0         1.0  
4      0.0         1.0         0.0         0.0  


In [5]:
# Vamos dividir a amostra em train e test:
SEED = 11
target_col = "cancel_sim"

X = tecx.drop(columns=[target_col])
y = tecx[target_col].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.60, random_state=SEED, stratify=y
)

print(X_train.shape, X_test.shape)
print("Proporção classe 1 (train/test):",
      y_train.mean().round(3), y_test.mean().round(3))

(1200, 11) (800, 11)
Proporção classe 1 (train/test): 0.238 0.239


In [7]:
# Treinar MLP de classificação
from sklearn.neural_network import MLPClassifier

SEED = 36

clf = MLPClassifier(
    hidden_layer_sizes=(6,),
    activation="relu",
    solver="lbfgs",
    alpha=0.0001,
    max_iter=2000,
    early_stopping=True,
    learning_rate_init=0.001,
    random_state=SEED
)

clf.fit(X_train, y_train)

# probabilidades no conjunto de teste 
pred = clf.predict_proba(X_test)

# probabilidade da classe positiva
psim = pred[:, 1]

# 3 primeiras linhas
print(pd.DataFrame(pred[:3], columns=['p(0)','p(1)']).to_string(index=False))

    p(0)     p(1)
0.006059 0.993941
0.494539 0.505461
0.992578 0.007422


In [9]:
# Métricas (com cutoff=0.5)
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score

threshold = 0.5
klass = (psim > threshold).astype(int)

auc = roc_auc_score(y_test, psim)
cm  = confusion_matrix(y_test, klass, labels=[0, 1])
acc = accuracy_score(y_test, klass)

print(f"AUC: {auc:.6f}")
print(pd.DataFrame(cm, index=['y_true 0', 'y_true 1'],
                      columns=['y_pred 0', 'y_pred 1']).to_string())
print(f"Accuracy: {acc:.5f}")

AUC: 0.855234
          y_pred 0  y_pred 1
y_true 0       569        40
y_true 1        98        93
Accuracy: 0.82750
